In [1]:
!pip install datasets

  Using cached datasets-3.2.0-py3-none-any.whl.metadata (20 kB)
  Using cached huggingface_hub-0.27.0-py3-none-any.whl.metadata (13 kB)
Using cached datasets-3.2.0-py3-none-any.whl (480 kB)
Using cached huggingface_hub-0.27.0-py3-none-any.whl (450 kB)


In [2]:
from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.hi")

In [3]:
ds = ds['train'].to_pandas()

In [4]:
ds.shape

(163093, 4)

In [5]:
ds.drop_duplicates(inplace=True)

In [6]:
ds.shape

(163093, 4)

In [7]:
ds

,id,url,title,text
0,10,https://hi.wikipedia.org/wiki/%E0%A4%B9%E0%A4%...,हम होंगे कामयाब,हम होंगे कामयाब ( का गिरिजा कुमार माथुर द्वारा...
1,14,https://hi.wikipedia.org/wiki/%E0%A4%A6%E0%A5%...,दैनिक पूजा,दैनिक पूजा विधि हिन्दू धर्म की कई उपासना पद्धत...
2,54,https://hi.wikipedia.org/wiki/%E0%A4%B9%E0%A4%...,हिन्दी,हिन्दी जिसके मानकीकृत रूप को मानक हिन्दी कहा ज...
3,56,https://hi.wikipedia.org/wiki/%E0%A4%AB%E0%A4%...,फ़ारसी भाषा,"फ़ारसी (), एक भाषा है जो ईरान, ताजिकिस्तान, अफ..."
4,58,https://hi.wikipedia.org/wiki/%E0%A4%A6%E0%A5%...,देवनागरी,देवनागरी भारतीय उपमहाद्वीप में प्रयुक्त प्राची...
...,...,...,...,...
163088,1492984,https://hi.wikipedia.org/wiki/%E0%A4%9F%E0%A5%...,टोनांतिंस,() ब्राज़ील के आमेज़ोनास राज्य का शहर है। इसकी...
163089,1492985,https://hi.wikipedia.org/wiki/%E0%A4%AC%E0%A4%...,"बार्सेलोस, आमेज़ोनास",बार्सेलोस () ब्राज़ील के आमेज़ोनास राज्य का शह...
163090,1493004,https://hi.wikipedia.org/wiki/%E0%A4%A6%20%E0%...,द नन (2018 फिल्म),द नन 2018 की अमेरिकी गॉथिक अलौकिक हॉरर फिल्म ह...
163091,1493015,https://hi.wikipedia.org/wiki/%E0%A4%B6%E0%A4%...,शहज़ाद शेख,शहज़ाद शेख एक भारतीय टेलीविजन अभिनेता हैं। उन...


In [8]:
ds.drop(['id', 'url'],axis=1, inplace=True)

In [9]:
ds

,title,text
0,हम होंगे कामयाब,हम होंगे कामयाब ( का गिरिजा कुमार माथुर द्वारा...
1,दैनिक पूजा,दैनिक पूजा विधि हिन्दू धर्म की कई उपासना पद्धत...
2,हिन्दी,हिन्दी जिसके मानकीकृत रूप को मानक हिन्दी कहा ज...
3,फ़ारसी भाषा,"फ़ारसी (), एक भाषा है जो ईरान, ताजिकिस्तान, अफ..."
4,देवनागरी,देवनागरी भारतीय उपमहाद्वीप में प्रयुक्त प्राची...
...,...,...
163088,टोनांतिंस,() ब्राज़ील के आमेज़ोनास राज्य का शहर है। इसकी...
163089,"बार्सेलोस, आमेज़ोनास",बार्सेलोस () ब्राज़ील के आमेज़ोनास राज्य का शह...
163090,द नन (2018 फिल्म),द नन 2018 की अमेरिकी गॉथिक अलौकिक हॉरर फिल्म ह...
163091,शहज़ाद शेख,शहज़ाद शेख एक भारतीय टेलीविजन अभिनेता हैं। उन...


In [10]:
ds.drop_duplicates(inplace=True)

In [11]:
ds.shape

(163093, 2)

In [13]:
!pip install datasketch

In [14]:
from datasketch.minhash import MinHash

In [15]:
ds['title_text'] = ds['title'] + ds['text']

In [ ]:
# for OOM error, i have taken 16000 rows

In [20]:
shingles = ds['title_text'][:16000].to_numpy()

In [30]:
signatures = []
cnt = 0
for i_doc in range(len(shingles)):
    cnt = cnt + 1;
    if cnt%1000 == 0:
        print(cnt)
    m = MinHash()
    for d in shingles[i_doc]:
        m.update(d.encode('utf8'))
    signatures.append(m)

index_keep = []    
for i_doc in range(len(shingles)-1):
    flag = True
    for j_doc in range(i_doc + 1, len(shingles)):
        jaccard_similarity = signatures[i_doc].jaccard(signatures[j_doc])
        if jaccard_similarity > 0.75:
            flag = False
            break
    if flag:
        index_keep.append(i_doc)

1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000


In [31]:
len(jac_sim)

127992000

In [36]:
shingles[0]

'हम होंगे कामयाबहम होंगे कामयाब ( का गिरिजा कुमार माथुर द्वारा किया गया हिंदी भावानुवाद) एक प्रतिरोध गीत है। यह गीत बीसवीं सदी में नागरिक अधिकार आंदोलन का प्रधान स्वर बना। इस गीत को आमतौर पर "I\'ll Overcome Some Day" ("आई विल ओवरकम सम डे") से काव्यावतरित माना जाता है, जो चार्ल्स अल्बर्ट टिंडले द्वारा गाया गया था और जिसे 1900 में पहली बार प्रकाशित किया गया था।\n\nसन्दर्भ\nHum Honge Kamyab Lyrics \nनागरिक अधिकार आंदोलन\nदेशभक्ति के गीत\nआधार'

In [37]:
words = []
for i in index_keep:
    words.append(shingles[i])

In [39]:
words

['उत्तर दिशाउत्तर दिशा मुख्य चार दिशाओं में से एक है। \n\nदिशाएँ\n\nda:Kompasretning#Nord भारत उपमहाद्वीप के उत्तर में हिमालय है',
 'वायरस(1) कम्प्यूटर वायरस\n\n(2) जैविक वायरस या विषाणु',
 'मईमई ग्रेगोरी कैलेंडर के वर्ष का पांचवां महीना है जिसमे 31 दिन होते हैं।\n\nदिवस\n 1 मई - मई दिवस, मजदूर दिवस।\n 22 मई - विश्व जैव विविधता दिवस\n\n \n8 may:world redcross day',
 'हिरोडोटसहिरोडोटस (ग्रीक: Ἡρόδοτος Ἁλικαρνᾱσσεύς Hērodotos Halikarnāsseus) (मृत्\u200dयु ४२५ ई.पू), यूनान के इतिहासकार एवं भूगोलवेत्ता थे। उन्होंने अपने इतिहास का विषय पेलोपोनेसियन युद्ध को बनाया था। उनके द्वारा लिखित पुस्तक हिस्टोरिका थी।\n\nभूगोलवेत्ता\nयूनानी भूगोलवेत्ता',
 'अम्मानअम्मान (अरबी عمان), जोर्डन राज्य की राजधानी, १२ लाख से ज़्यादा लोगों का शहर है।\n\nबाहरी कडियाँ \n अल अह्लिय्या अम्मान विश्वविद्यालय जालस्थल - अंग्रेज़ी में अम्मान के बारे में जानकारी\n\nएशियाई शहर\nएशिया में राजधानियाँ',
 'दीपावलीदीपावली ( = दीप + आवलिः = पंक्ति, अर्थात् पंक्ति में रखे हुए दीपक) शरदृतु (उत्तरी गोलार्ध) में प्रतिवर्ष मनाया जाने

In [40]:
import re
from collections import defaultdict

def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i],symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

def get_vocab(data):
    vocab = defaultdict(int)
    for line in data:
        for word in line.split():
            vocab[' '.join(list(word)) + ' </w>'] += 1
    return vocab

def byte_pair_encoding(data, n):
    vocab = get_vocab(data)
    for i in range(n):
        pairs = get_stats(vocab)
        best = max(pairs, key=pairs.get)
        vocab = merge_vocab(best, vocab)
    return vocab

# Example usage:
data = words

n = 230
bpe_pairs = byte_pair_encoding(data, n)
bpe_pairs

{'उत् त र</w>': 91,
 'दि शा उत् त र</w>': 1,
 'दि श ा</w>': 21,
 'मु ख ्य</w>': 93,
 'चा र</w>': 50,
 'दि शा ओं</w>': 1,
 'में</w>': 5163,
 'से</w>': 2714,
 'एक</w>': 1924,
 'है।</w>': 2364,
 'दि शा ए ँ</w>': 2,
 'd a : K o m p a s r e t n in g # N or d</w>': 1,
 'भार त</w>': 351,
 'उ प म हा द्व ी प</w>': 17,
 'के</w>': 8115,
 'हि मा ल य</w>': 13,
 'है</w>': 1358,
 'वा य र स ( 1 )</w>': 1,
 'क म् प ्य ू ट र</w>': 8,
 'वा य र स</w>': 5,
 '( 2 )</w>': 6,
 'ज ै वि क</w>': 6,
 'या</w>': 541,
 'वि ष ा ण ु</w>': 1,
 'म ई म ई</w>': 1,
 'ग ्रे ग ो री</w>': 20,
 'क ै ले ंड र</w>': 68,
 'वर् ष </w>': 184,
 'का</w>': 2795,
 'पा ं च वा ं</w>': 5,
 'म ह ी ना</w>': 5,
 'जि सम े</w>': 5,
 '3 1 </w>': 7,
 'दि न</w>': 197,
 'हो ते</w>': 110,
 'हैं।</w>': 868,
 'दि व स</w>': 19,
 '1 </w>': 53,
 'म ई</w>': 33,
 '-</w>': 449,
 'दि व स ,</w>': 1,
 'म ज द ू र</w>': 3,
 'दि व स ।</w>': 1,
 '2 2 </w>': 14,
 'वि श् व</w>': 127,
 'ज ै व</w>': 3,
 'वि वि ध ता</w>': 7,
 '8 </w>': 28,
 'm a y : w or l d</w>': 1,
 